# Prompting Foundations

Prompt engineering is the practice of turning a fuzzy human request into a precise contract that a language model can follow. In this notebook you will build that contract explicitly, then send both zero-shot and few-shot prompts to a real provider.

The repo uses Groq as the primary route and Mistral as the alternative route. Both are called through the [OpenAI Python library](https://github.com/openai/openai-python), which keeps the lesson focused on prompts instead of provider-specific SDK details.


## Learning Objectives

By the end of this notebook, you should be able to:

- separate role, task, context, constraints, examples, and output format,
- render prompt templates without hiding important variables,
- run a zero-shot and a few-shot prompt against Groq or Mistral,
- compare real responses with a small review checklist,
- explain when a prompt needs more context versus more examples.


## Model And Prompt Context

Prompting sits on top of model behavior. The Transformer diagram gives you a mental model for why context, examples, and output constraints all enter the same sequence-processing system. The important idea is not to memorize every block, but to notice that input tokens, positional information, attention, and output probabilities are part of one pipeline.

![Transformer model architecture](assets/01-transformer-architecture.png)

Source: [Wikimedia Commons - The Transformer model architecture](https://commons.wikimedia.org/wiki/File:The-Transformer-model-architecture.png)

The training workflow below helps separate what you can change with a prompt from what belongs to model training or alignment. In this repository, you work mostly at inference time: you change instructions, context, examples, and output contracts without retraining the model.

![Three-stage LLM training workflow](assets/01-llm-training-workflow.png)

Source: [Wikimedia Commons - Three-stage large language model training workflow](https://commons.wikimedia.org/wiki/File:Three-stage_large_language_model_training_workflow.svg)


## Provider Setup

The notebooks read provider settings from `.env`:

```text
LLM_PROVIDER=groq
GROQ_API_KEY=...
MISTRAL_API_KEY=...
```

Groq is [OpenAI-compatible](https://console.groq.com/docs/openai) through `https://api.groq.com/openai/v1`. Mistral also exposes chat completions at `https://api.mistral.ai/v1`. If no key is present, the helper below enters dry-run mode so the notebook still executes and shows where the model call would happen.


## Shared Provider Helper

Provider selection, OpenAI-compatible client creation, and dry-run chat behavior live in `prompt_helpers.py`. Keeping that plumbing in one place avoids repeating the same setup in every notebook while leaving the prompt, schema, guardrail, and evaluation logic visible here.


In [ ]:
import json

from prompt_helpers import chat, client_and_model

In [ ]:
client, model, provider, dry_reason = client_and_model()
print(f"Provider: {provider.name}")
print(f"Model: {model}")
print(dry_reason or "Live mode: API key found.")

## Prompt Anatomy

A useful prompt usually contains six pieces. Not every task needs all six, but when outputs are unreliable, this checklist helps you decide what to add.

| Part | Purpose | Common failure when missing |
| --- | --- | --- |
| Role | Sets the perspective or expertise | Output uses the wrong tone or assumptions |
| Task | Names the work to perform | Model explains instead of doing |
| Context | Supplies facts the model should use | Model fills gaps with guesses |
| Constraints | Defines boundaries and refusals | Output is too broad, unsafe, or off-topic |
| Examples | Shows the pattern | Model invents labels or format |
| Output format | Makes the result usable downstream | Output is hard to parse or compare |


In [ ]:
from string import Template

prompt_template = Template(
    """You are a support operations analyst.

Task: classify the customer message into one product-support category.

Context:
- Categories: BILLING, ACCOUNT_ACCESS, BUG_REPORT, FEATURE_REQUEST, OTHER
- Use OTHER when the message is not about product support.

Constraints:
- Return exactly one category.
- Do not explain the answer.

Customer message:
$message
"""
)

message = "I changed phones and cannot log in because the authenticator app is gone."
rendered_prompt = prompt_template.substitute(message=message)
print(rendered_prompt)

The template keeps the input variable visible. This matters because prompt bugs often come from mixing instructions and user content, or from accidentally dropping the task context when the prompt is reused.


In [ ]:
zero_shot_messages = [
    {"role": "system", "content": "You follow the user's output contract exactly."},
    {"role": "user", "content": rendered_prompt},
]

zero_shot_response = chat(zero_shot_messages, temperature=0.1)
print(zero_shot_response)

## From Zero-Shot To Few-Shot

Zero-shot prompting asks the model to infer the pattern from instructions only. Few-shot prompting adds examples. Examples are useful when labels are ambiguous, the output format is strict, or the task has local business rules that are not obvious from the label names.


In [ ]:
few_shot_examples = [
    ("My invoice has the wrong company address.", "BILLING"),
    ("The export button returns a 500 error.", "BUG_REPORT"),
    ("Can you add dark mode to the dashboard?", "FEATURE_REQUEST"),
]

example_block = "\n\n".join(
    f'Message: "{example}"\nCategory: {label}' for example, label in few_shot_examples
)
print(example_block)

The examples are now data, not hardcoded prose. That makes it easier to add, remove, or review examples without touching the rest of the prompt.



In [ ]:
few_shot_template = Template(
    """You are a support operations analyst.

Task: classify the customer message into one product-support category.

Categories:
- BILLING: invoices, payment methods, charges, refunds
- ACCOUNT_ACCESS: login, passwords, MFA, locked accounts
- BUG_REPORT: broken behavior, errors, regressions
- FEATURE_REQUEST: requests for new or changed product behavior
- OTHER: anything outside product support

Examples:
$examples

Return exactly one category.

Message: "$message"
Category:"""
)

few_shot_prompt = few_shot_template.substitute(message=message, examples=example_block)
print(few_shot_prompt)

In [ ]:
few_shot_messages = [
    {
        "role": "system",
        "content": "You classify support messages using the provided examples.",
    },
    {"role": "user", "content": few_shot_prompt},
]

few_shot_response = chat(few_shot_messages, temperature=0.1)
print(few_shot_response)

## Compare The Responses

A prompt is not better because it is longer. It is better when it produces outputs that are more correct, more stable, and easier to use. The table below makes the review explicit.


In [ ]:
def review_category_response(
    response: str, allowed_labels: set[str]
) -> dict[str, object]:
    """Check whether a classification response can be consumed by downstream code."""

    normalized = response.strip().splitlines()[0].strip().upper()
    return {
        "raw": response,
        "normalized_first_line": normalized,
        "is_allowed_label": normalized in allowed_labels,
        "is_single_token": len(normalized.split()) == 1,
    }


allowed = {"BILLING", "ACCOUNT_ACCESS", "BUG_REPORT", "FEATURE_REQUEST", "OTHER"}
comparison = {
    "zero_shot": review_category_response(zero_shot_response, allowed),
    "few_shot": review_category_response(few_shot_response, allowed),
}
print(json.dumps(comparison, indent=2))

## Prompt Debugging Moves

When a model response is wrong, avoid blindly adding more instructions. Choose the smallest change that targets the observed failure:

| Failure | Better first move |
| --- | --- |
| Wrong label | clarify label definitions or add a close example |
| Extra explanation | tighten the output contract |
| Hallucinated facts | add source/context boundaries |
| Unsafe answer | add guardrails before and after the call |
| Inconsistent style | add examples and lower temperature |


In [ ]:
def build_classification_prompt(
    message: str, examples: list[tuple[str, str]] | None = None
) -> str:
    """Create a compact prompt while keeping examples optional and readable."""

    examples = examples or []
    example_block = "\n\n".join(
        f'Message: "{example}"\nCategory: {label}' for example, label in examples
    )
    if example_block:
        example_block = f"\n\nExamples:\n{example_block}"

    return f"""Classify the support message into exactly one category.

Categories: BILLING, ACCOUNT_ACCESS, BUG_REPORT, FEATURE_REQUEST, OTHER{example_block}

Return only the category label.

Message: "{message}"
Category:"""


prompt = build_classification_prompt(
    "I was charged twice after upgrading my workspace.",
    examples=[("I cannot reset my password.", "ACCOUNT_ACCESS")],
)
print(prompt)

## Mini Exercise

Pick two ambiguous support messages from your own experience. For each one:

1. write a zero-shot prompt;
2. add one example that should disambiguate the task;
3. run both prompts;
4. record whether the example changed correctness, format, or confidence.

Keep the final answer concise: one sentence explaining which prompt you would ship and why.


## Correction

Here is one possible comparison. The messages are deliberately ambiguous: a refund combines product access and billing, while an authentication change could be mistaken for a general feature request. The examples make the desired labels explicit.


In [ ]:
solution_cases = [
    ("I cancelled yesterday but I was charged for next month.", "BILLING"),
    ("I need a new login code after replacing my phone.", "ACCOUNT_ACCESS"),
]
solution_examples = [
    ("I need a refund for a renewal charge.", "BILLING"),
    ("My authenticator no longer works.", "ACCOUNT_ACCESS"),
]

for support_message, expected in solution_cases:
    zero_prompt = build_classification_prompt(support_message)
    few_prompt = build_classification_prompt(support_message, solution_examples)
    zero_answer = chat([{"role": "user", "content": zero_prompt}], temperature=0.1)
    few_answer = chat([{"role": "user", "content": few_prompt}], temperature=0.1)
    print({"message": support_message, "expected": expected})
    print({"zero_shot": zero_answer, "few_shot": few_answer})

**Reference answer:** I would ship the few-shot prompt first because its examples clarify two easy-to-confuse label boundaries while preserving the one-label output contract. In dry-run mode, the code demonstrates the workflow; configure a provider key to compare real responses.


## Further Reading

- [Groq API reference](https://console.groq.com/docs/api-reference)
- [OpenAI prompting guide](https://platform.openai.com/docs/guides/prompting)
- [Groq prompt basics](https://console.groq.com/docs/prompting)
- [Mistral prompting guide](https://docs.mistral.ai/guides/prompting_capabilities/)
- [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165)
- [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)
